<a href="https://colab.research.google.com/github/pedroantoniolli-ops/Infnet/blob/main/PD_PedroAntoniolli_Sistemas_Cognitivos_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Projeto Disciplina** Sistemas Cognitivos com LLM (26E2_3)


**Aluno:** Pedro Domingos Antoniolli

**Professor:** Dr. Fernando Guimaraes Ferreira

**Link do Google Colab**:

**Titulo:** Aplicacao Funcional Cognitiva com LLM em Phyton

 **Definição do Problema**

 **Cenário:** Assistente Inteligente de Suporte Técnico para Proprietários e Mecânicos da motocicleta BMW F 800 GS.

 **Objetivo:** Responder a dúvidas complexas sobre manutenção, painel digital TFT, calibração e segurança do veículo, eliminando alucinações de modelos genéricos através do fornecimento de contexto preciso.


**Corpus utilizado:** Manual Tecnico da Motocicleta BMW F800GS, sendo este um documento estruturado em texto, em pdf, contendo capítulos técnicos específicos, tabelas de torque e procedimentos de oficina.

**Justificativas:**

Essa escolha melhora a qualidade do *case* pelos seguintes motivos:
*   Se encontra em portugues
*   Se trata de um manual tecnico
*   Desafio real de ingestao, uma vez que manuais automotivos contêm tabelas (ex: calibragem de pneus) e termos técnicos complexos, perfeitos para testar a qualidade do seu algoritmo de extração de texto
*   Avaliacao precisa, ou seja, como o manual contém dados muito específicos (ex: torque exato de parafusos ou especificações do óleo), fica fácil validar se o LLM está respondendo com base no contexto ou alucinando.
*   Possui valor pratico, ja que o sistema pode se transformar em um assistente real para motociclistas ou mecanicos.
*   A base eh composta por um arquivo pdf de 293, que eh suficiente para permitir consultas, recuperação de contexto, comparação de respostas e avaliação das estratégias de busca.

Tal cenário permite avaliar tanto o uso de LLMs quanto a arquitetura do sistema construído.








**Configuracao do Google Colab**

Obs: Inicialmente foi alterado o ambiente de execucao para uso de GPU (T4GPU) no opcao Runtime Environment.

A ativação da GPU no Google Colab é um processo simples que pode acelerar drasticamente o treinamento de modelos de Inteligência Artificial e processamento de dados.

***Etapa 1 - Aplicacao NLP com Modelos Pre Treinados***

**1. Instalar Bibliotecas Necessarias**

In [4]:
!pip install -q pdfplumber
!pip install -q sentence-transformers
!pip install -q tokenizer
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q huggingface-hub --upgrade # Explicitly upgrade huggingface-hub
!pip install pypdf
!pip install -q transformers
!pip install -q torch
!pip install -q faiss-cpu
!pip install -q accelerate
!pip install -q pymupdf
!pip install -q pydantic
!pip install -q pandas
!pip install --force-reinstall Pillow

import transformers
import tokenizer
import langchain
import torch

# Define DEVICE after torch import
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("versao do Transformer")
print(transformers.__version__)
print("versao do Tokenizer")
print(tokenizer.__version__)
print("versao do Langchain")
print(langchain.__version__)
print("versao do Torch")
print(torch.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 123.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into

**2. Fazer Upload do Manual da F800GS - Arquivo pdf**

In [5]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving Manual_BMW_F800GS_PORT.pdf to Manual_BMW_F800GS_PORT.pdf
User uploaded file "Manual_BMW_F800GS_PORT.pdf" with length 4920806 bytes


In [6]:
import pdfplumber

texto = ""

with pdfplumber.open("/content/Manual_BMW_F800GS_PORT.pdf") as pdf:

    print(f"Número de páginas: {len(pdf.pages)}")

    for pagina in pdf.pages:

        conteudo = pagina.extract_text()

        if conteudo:
            texto += conteudo + "\n"
        else:
            print("O arquivo PDF nao foi carregado. Por favor, faca o upload usando `files.upload()`")

print(texto[:1000])

Número de páginas: 293
BMW
MOTORRAD
INSTRUÇÕES DE
OPERAÇÃO
F 800 GS
MAKELIFEARIDE
Dadosdoveículo
Modelo
Númerodeidentificaçãodoveículo
Códigodacor
Primeiramatriculação
Chapadamatrícula
Dadosdoconcessionário
FuncionáriodoServiço
Senhora/Senhor
Númerodetelefone
Endereçodoconcessionário/telefone(carimbodaempresa)
A SUA BMW.
Agradecemosasuapreferênciaporumveículoda
BMWMotorradedamos-lheasboas-vindasaocírculode
condutoresBMW.Familiarize-secomoseunovoveículo,para
quepossamovimentar-secomsegurançanotrânsito.
Sobreestasinstruçõesdeoperação
Leiaestasinstruçõesdeutilização,antesdecolocarasuanova
BMWemmarcha. Aquiencontraráindicaçõesimportantesrelati-
vamenteàoperaçãodoveículo,quelhepermitirãoaproveitarao
máximotodasasvantagenstécnicasdasuaBMW.
Paraalémdisso,obtéminformaçõesrelativasàmanutençãoe
conservação,quecontribuemparaasegurançadefuncionamento
enaestrada,assimcomoparaapreservaçãodovalordoseuveí-
culo.
SenofuturopretendervenderasuaBMW,lembre-sedeentregar
tambémasinstruçõesdeutilização. Sãoum

**3. Limpeza**

In [7]:
import re

texto = re.sub(r'\n+', '\n', texto)

texto = re.sub(r'Página\s+\d+', '', texto)

texto = re.sub(r'\s+', ' ', texto)

print(texto[:2000])

BMW MOTORRAD INSTRUÇÕES DE OPERAÇÃO F 800 GS MAKELIFEARIDE Dadosdoveículo Modelo Númerodeidentificaçãodoveículo Códigodacor Primeiramatriculação Chapadamatrícula Dadosdoconcessionário FuncionáriodoServiço Senhora/Senhor Númerodetelefone Endereçodoconcessionário/telefone(carimbodaempresa) A SUA BMW. Agradecemosasuapreferênciaporumveículoda BMWMotorradedamos-lheasboas-vindasaocírculode condutoresBMW.Familiarize-secomoseunovoveículo,para quepossamovimentar-secomsegurançanotrânsito. Sobreestasinstruçõesdeoperação Leiaestasinstruçõesdeutilização,antesdecolocarasuanova BMWemmarcha. Aquiencontraráindicaçõesimportantesrelati- vamenteàoperaçãodoveículo,quelhepermitirãoaproveitarao máximotodasasvantagenstécnicasdasuaBMW. Paraalémdisso,obtéminformaçõesrelativasàmanutençãoe conservação,quecontribuemparaasegurançadefuncionamento enaestrada,assimcomoparaapreservaçãodovalordoseuveí- culo. SenofuturopretendervenderasuaBMW,lembre-sedeentregar tambémasinstruçõesdeutilização. Sãoumaparteimportante doseuv

**4. Chunking (Dividir em Blocos)**

In [8]:
from textwrap import wrap

chunks = wrap(texto,600)

print("Quantidade de chunks:",len(chunks))

print(chunks[10])

Quantidade de chunks: 408
deriscoelevado. Ainob- servânciadáorigemamorte ouferimentosgraves. 5 SA Equipamentoespecial. EQUIPAMENTO Osequipamentosex- traBMWMotorradjá Aocomprarasua sãomontadosdurante BMWMotorrad,decidiu- aproduçãodosveícu- seporummodelocomum los. equipamentoindividual. Estasinstruçõesdeutilização SZ Equipamentoextra. descrevemosequipamentos Oequipamentoextra opcionais(SA)disponibilizados BMWMotorradpode pelaBMWeequipamentoextra seradquiridoeree- (SZ)selecionado. Pedimosa quipadoatravésdo suacompreensãoparaofacto seuconcessionário detambémestaremdescritas BMWMotorrad. versõesdoequipamentoque, ABS


**5. Tokenizar**

Usando Modelo Encoder do Hugging Face

O modelo escolhido foi o all-MiniLM-L6-v2, por apresentar as seguintes caracteristicas:

Arquitetura: Encoder-only

Dimensao: 384

Tamanho: 22 milhoes de parametros

Uso: Busca Semantica

Linguagem: Multilingue razoavel

Velocidade: Muito Alta

**Justificativas:**

Gratuito.

Compativel com o Google Colab

Excelente desempenho para busca semantica

Amplamente utilizado em sistemas RAG



In [9]:
from transformers import AutoTokenizer

modelo = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(modelo)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Exemplo do Texto do Manual BMW

In [10]:
texto = """
Verifique o nível do óleo do motor antes da utilização da motocicleta.
A motocicleta deve estar em posição vertical e o motor deve estar aquecido.
"""

Tokenizacao Simples

In [11]:
tokens = tokenizer.tokenize(texto)

print(tokens)


['ve', '##ri', '##fi', '##que', 'o', 'ni', '##vel', 'do', 'ole', '##o', 'do', 'motor', 'ant', '##es', 'da', 'ut', '##ili', '##za', '##cao', 'da', 'mo', '##to', '##cic', '##let', '##a', '.', 'a', 'mo', '##to', '##cic', '##let', '##a', 'dev', '##e', 'est', '##ar', 'em', 'po', '##sic', '##ao', 'vertical', 'e', 'o', 'motor', 'dev', '##e', 'est', '##ar', 'a', '##que', '##ci', '##do', '.']


**Converter os Tokens para Strings**

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texto = """
Verifique o nível do óleo do motor da BMW F800GS.
"""

tokens = tokenizer.tokenize(texto)

print("Tokens originais:")
print(tokens)


texto_reconstruido = tokenizer.convert_tokens_to_string(tokens)

print("\nTexto reconstruído:")
print(texto_reconstruido)

Tokens originais:
['ve', '##ri', '##fi', '##que', 'o', 'ni', '##vel', 'do', 'ole', '##o', 'do', 'motor', 'da', 'bmw', 'f', '##80', '##0', '##gs', '.']

Texto reconstruído:
verifique o nivel do oleo do motor da bmw f800gs.


**6. Converter Tokens para IDs**

Uma vez que os modelos nao trabalham com palavras, somente com numeros

In [13]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[2310, 3089, 8873, 4226, 1051, 9152, 15985, 2079, 15589, 2080, 2079, 5013, 4830, 13154, 1042, 17914, 2692, 5620, 1012]


**7. Tokenizacao completa para o modelo**



In [14]:
entrada = tokenizer(

    texto,

    return_tensors="pt",

    padding=True,

    truncation=True,

    max_length=512

)

print(entrada)

{'input_ids': tensor([[  101,  2310,  3089,  8873,  4226,  1051,  9152, 15985,  2079, 15589,
          2080,  2079,  5013,  4830, 13154,  1042, 17914,  2692,  5620,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


**8. Visualizar Tokens e IDs**



In [15]:
for token,id in zip(

    tokens,

    ids

):

    print(token," -> ",id)

ve  ->  2310
##ri  ->  3089
##fi  ->  8873
##que  ->  4226
o  ->  1051
ni  ->  9152
##vel  ->  15985
do  ->  2079
ole  ->  15589
##o  ->  2080
do  ->  2079
motor  ->  5013
da  ->  4830
bmw  ->  13154
f  ->  1042
##80  ->  17914
##0  ->  2692
##gs  ->  5620
.  ->  1012


**9. Verificar quantidade de Tokens**

Importante para RAG e Chunking

In [16]:
quantidade = len(

    tokenizer.encode(texto)

)

print(
    "Quantidade de tokens:",
    quantidade
)

Quantidade de tokens: 21


**10. Tokenizar varios chunks do PDF**

No Pipeline RAG

In [17]:
for i,chunk in enumerate(chunks[:5]):

    tokens = tokenizer.encode(
        chunk,
        truncation=True,
        max_length=512
    )

    print(
        "Chunk:",
        i,
        "Tokens:",
        len(tokens)
    )

Chunk: 0 Tokens: 217
Chunk: 1 Tokens: 234
Chunk: 2 Tokens: 207
Chunk: 3 Tokens: 219
Chunk: 4 Tokens: 205


**11. Ver o Limite do Modelo**



In [18]:
print(tokenizer.encode)
print(tokenizer.model_max_length)

<bound method PreTrainedTokenizerBase.encode of BertTokenizer(name_or_path='sentence-transformers/all-MiniLM-L6-v2', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})>
512


A tokenização foi realizada utilizando o tokenizer disponibilizado pelo Hugging Face Transformers.

O processo converte os textos extraídos do manual BMW F800GS em sequências de tokens e posteriormente em identificadores numéricos utilizados pelo modelo de linguagem.

Foi aplicado truncamento com limite de 512 tokens, compatível com a arquitetura do modelo escolhido.

Essa etapa se torna fundamental para controlar o tamanho das entradas e evitar exceder a janela de contexto do modelo durante a geração dos embeddings e respostas.

**12. Criar Indice Vetorial**

In [19]:
print(len(tokenizer))


30522


Funcao para Gerar os Embeddings

Reinstalar o Pillow para evitar conflito

In [20]:
!pip uninstall -y Pillow
!pip install --no-cache-dir --force-reinstall Pillow==11.3.0

Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 104.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.


Verificar a versao do Pillow

In [21]:
import PIL

print("Versão Pillow:", PIL.__version__)
print("Local:", PIL.__file__)

Versão Pillow: 11.3.0
Local: /usr/local/lib/python3.12/dist-packages/PIL/__init__.py


Testar o Sentence Transformer

In [22]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDING = "intfloat/multilingual-e5-base"

modelo_embeddings = SentenceTransformer(
    MODELO_EMBEDDING
)

print("Modelo carregado com sucesso")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo carregado com sucesso


Criar a funcao do Embedding

In [23]:
import numpy as np

def gerar_embeddings(textos, tipo="passage"):
    """
    Gera embeddings normalizados usando multilingual-e5-base.

    tipo:
      passage = documentos/chunks
      query   = perguntas
    """

    if isinstance(textos, str):
        textos = [textos]

    textos_formatados = [
        f"{tipo}: {texto}"
        for texto in textos
    ]

    vetores = modelo_embeddings.encode(
        textos_formatados,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    return vetores.astype("float32")

Transformar os chunks em lista de texto

In [24]:
chunks[0]

'BMW MOTORRAD INSTRUÇÕES DE OPERAÇÃO F 800 GS MAKELIFEARIDE Dadosdoveículo Modelo Númerodeidentificaçãodoveículo Códigodacor Primeiramatriculação Chapadamatrícula Dadosdoconcessionário FuncionáriodoServiço Senhora/Senhor Númerodetelefone Endereçodoconcessionário/telefone(carimbodaempresa) A SUA BMW. Agradecemosasuapreferênciaporumveículoda BMWMotorradedamos-lheasboas-vindasaocírculode condutoresBMW.Familiarize-secomoseunovoveículo,para quepossamovimentar-secomsegurançanotrânsito. Sobreestasinstruçõesdeoperação Leiaestasinstruçõesdeutilização,antesdecolocarasuanova BMWemmarcha.'

In [25]:
print(type(chunks))
print(type(chunks[0]))

print("\nPrimeiro chunk:")
print(chunks[0][:500])

<class 'list'>
<class 'str'>

Primeiro chunk:
BMW MOTORRAD INSTRUÇÕES DE OPERAÇÃO F 800 GS MAKELIFEARIDE Dadosdoveículo Modelo Númerodeidentificaçãodoveículo Códigodacor Primeiramatriculação Chapadamatrícula Dadosdoconcessionário FuncionáriodoServiço Senhora/Senhor Númerodetelefone Endereçodoconcessionário/telefone(carimbodaempresa) A SUA BMW. Agradecemosasuapreferênciaporumveículoda BMWMotorradedamos-lheasboas-vindasaocírculode condutoresBMW.Familiarize-secomoseunovoveículo,para quepossamovimentar-secomsegurançanotrânsito. Sobreestasinstru


In [26]:
textos_chunks = chunks

print("Quantidade de chunks:", len(textos_chunks))
print("\nPrimeiro chunk:")
print(textos_chunks[0][:500])

Quantidade de chunks: 408

Primeiro chunk:
BMW MOTORRAD INSTRUÇÕES DE OPERAÇÃO F 800 GS MAKELIFEARIDE Dadosdoveículo Modelo Númerodeidentificaçãodoveículo Códigodacor Primeiramatriculação Chapadamatrícula Dadosdoconcessionário FuncionáriodoServiço Senhora/Senhor Númerodetelefone Endereçodoconcessionário/telefone(carimbodaempresa) A SUA BMW. Agradecemosasuapreferênciaporumveículoda BMWMotorradedamos-lheasboas-vindasaocírculode condutoresBMW.Familiarize-secomoseunovoveículo,para quepossamovimentar-secomsegurançanotrânsito. Sobreestasinstru


**Gerar Embeddings**

In [27]:
embeddings = gerar_embeddings(
    textos_chunks,
    tipo="passage"
)

print("Tipo:", type(embeddings))
print("Shape:", embeddings.shape)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Tipo: <class 'numpy.ndarray'>
Shape: (408, 768)


**Estrutura**

chunks (list de strings) => gerar_embeddings() => multilingual-e5-base => numpy.ndarray => (N chunks × 768 dimensões)

**Criar o Indice FAISS**



In [28]:
import faiss

dimensao = embeddings.shape[1]

indice = faiss.IndexFlatIP(dimensao)

indice.add(embeddings)

print("Dimensão dos embeddings:", dimensao)
print("Quantidade de vetores indexados:", indice.ntotal)

Dimensão dos embeddings: 768
Quantidade de vetores indexados: 408


Como foram normalizados os embeddings, IndexFlatIP pode ser usado para comparar os vetores de forma equivalente à similaridade de cosseno.

**Criar a Funcao de Busca Semantica**

Como os chunks gerados corresponde a uma lista de strings


In [29]:
def buscar_semanticamente(pergunta, top_k=5):

    embedding_pergunta = gerar_embeddings(
        pergunta,
        tipo="query"
    )

    scores, indices = indice.search(
        embedding_pergunta,
        top_k
    )

    resultados = []

    for score, idx in zip(scores[0], indices[0]):

        resultados.append({
            "score": float(score),
            "chunk_id": int(idx),
            "texto": chunks[idx]
        })

    return resultados

**Fazer a primeira consulta**

Como verificar o nivel do oleo

In [30]:
pergunta = "Como verificar o nível do óleo do motor?"

resultados = buscar_semanticamente(
    pergunta,
    top_k=5
)

for posicao, resultado in enumerate(resultados, start=1):

    print("=" * 100)

    print(
        f"TOP {posicao} | "
        f"Score: {resultado['score']:.4f} | "
        f"Chunk: {resultado['chunk_id']}"
    )

    print()

    print(resultado["texto"][:800])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

TOP 1 | Score: 0.8563 | Chunk: 381

Verificaroníveldoóleodotravãodianteiroetraseiro Inspeçãovisualdostubosdotravão,tubosflexíveisdotravãoe ligações Verificarapressãoeaprofundidadedeperfildospneus Verificarelubrificaroacionamentodecorrente Verificarasuavidadedemovimentododescansolateral Verificarodescansoarticuladoemrelaçãoasuavidadedemo- vimento Verificaroapoiosuperiordadireção Verificarailuminaçãoeosistemadesinalização Testedefuncionamento,inibiçãodoarranquedomotor Inspeçãofinaleverificaçãodasegurançanaestrada Efetuarotesteaoveículoatravésdosistemadediagnóstico BMWMotorrad Definiradatadoserviçoeadistânciaremanescentecomosis-
TOP 2 | Score: 0.8550 | Chunk: 379

BMWMotorrad Mudançadeóleonomotorcomconversãodefiltro(filtrodo óleocomprido) Ajustaroapoiosuperiordadireção Verificaroníveldolíquidoderefrigeração Verificaroníveldoóleodotravãodianteiro Verificaroníveldoóleodotravãotraseiro Verificar/ajustarafolgadaembraiagem Verificaraflechadacorrenteelubrificaracorrentedetransmis- são Verificar

Esses resultados mostram que os embeddings estão funcionando corretamente. A recuperação encontrou trechos relacionados à manutenção e ao nível de óleo.

**Análise dos resultados**

TOP 1 (0,8563): checklist de manutenção com "Verificar o nível do óleo..." → muito relevante.

TOP 2 (0,8550): procedimentos de manutenção incluindo óleo do motor e outros itens → muito relevante.

TOP 3 (0,8515): procedimento detalhado para completar e verificar o óleo do motor → o mais útil para responder à pergunta, embora tenha score ligeiramente menor.

**O que foi feito ate aqui**

Manual BMW F800GS => Extração do texto => Chunking => multilingual-e5-base => Embeddings => FAISS => Pergunta => Embedding da pergunta => Similaridade de cosseno => TOP 5 chunks

O texto esta sem espacos, o que pode prejudicar o RAG. Acao: corrigir o texto

**Recriar os chunks**

In [31]:
import fitz

def extrair_texto_pdf(caminho_pdf):

    documento = fitz.open(caminho_pdf)

    paginas = []

    for numero_pagina, pagina in enumerate(documento):

        palavras = pagina.get_text("words")

        palavras = sorted(
            palavras,
            key=lambda x: (round(x[1], 1), x[0])
        )

        texto = " ".join(
            palavra[4]
            for palavra in palavras
        )

        paginas.append({
            "pagina": numero_pagina + 1,
            "texto": texto
        })

    documento.close()

    return paginas

**Melhorar a extracao do PDF**

Como esta sendo usado PyMuPDF, sera refeita a extração usando palavras e coordenadas

In [32]:
import os

arquivos_pdf = [
    arquivo
    for arquivo in os.listdir()
    if arquivo.lower().endswith(".pdf")
]

print("PDFs encontrados:", arquivos_pdf)

if len(arquivos_pdf) == 0:
    raise FileNotFoundError(
        "Nenhum arquivo PDF encontrado no Colab"
    )

ARQUIVO_PDF = arquivos_pdf[0]

print("PDF utilizado:", ARQUIVO_PDF)

paginas = extrair_texto_pdf(ARQUIVO_PDF)

print("Quantidade de páginas:", len(paginas))

PDFs encontrados: ['Manual_BMW_F800GS_PORT.pdf']
PDF utilizado: Manual_BMW_F800GS_PORT.pdf
Quantidade de páginas: 293


**Validar a Extracao**

In [33]:
print(type(paginas))
print(type(paginas[0]))

print("\nPágina 1:")
print(paginas[0]["texto"][:1000])

<class 'list'>
<class 'dict'>

Página 1:
BMW MOTORRAD INSTRUÇÕES DE OPERAÇÃO F 800 GS MAKE LIFE A RIDE


**Recriar os chunks**

In [34]:
def criar_chunks(paginas, tamanho=1000, sobreposicao=200):

    chunks = []

    for pagina in paginas:

        texto = pagina["texto"]

        inicio = 0

        while inicio < len(texto):

            fim = inicio + tamanho

            trecho = texto[inicio:fim].strip()

            if len(trecho) > 100:

                chunks.append({
                    "pagina": pagina["pagina"],
                    "texto": trecho
                })

            inicio += tamanho - sobreposicao

    return chunks


chunks = criar_chunks(paginas)

print("Quantidade de chunks:", len(chunks))

print("\nPrimeiro chunk:")
print(chunks[0])

Quantidade de chunks: 450

Primeiro chunk:
{'pagina': 2, 'texto': 'Dados do veículo Modelo Número de identificação do veículo Código da cor Primeira matriculação Chapa da matrícula Dados do concessionário Funcionário do Serviço Senhora/Senhor Número de telefone Endereço do concessionário/telefone (carimbo da empresa)'}


In [35]:
print(chunks[0]["pagina"])
print(chunks[0]["texto"][:500])

2
Dados do veículo Modelo Número de identificação do veículo Código da cor Primeira matriculação Chapa da matrícula Dados do concessionário Funcionário do Serviço Senhora/Senhor Número de telefone Endereço do concessionário/telefone (carimbo da empresa)


Agora o problema dos espacos foi corrigido

Proximo passo: **Recriar os embeddings**

Preparando os textos

In [36]:
textos_chunks = [
    chunk["texto"]
    for chunk in chunks
]

print("Quantidade de textos:", len(textos_chunks))

print("\nExemplo:")
print(textos_chunks[0][:500])

Quantidade de textos: 450

Exemplo:
Dados do veículo Modelo Número de identificação do veículo Código da cor Primeira matriculação Chapa da matrícula Dados do concessionário Funcionário do Serviço Senhora/Senhor Número de telefone Endereço do concessionário/telefone (carimbo da empresa)


**Gerar novamente os embeddings**

Agora chunks é uma lista de dicionários:

In [37]:
embeddings = gerar_embeddings(
    textos_chunks,
    tipo="passage"
)

print("\nEmbeddings gerados")

print("Tipo:", type(embeddings))
print("Shape:", embeddings.shape)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


Embeddings gerados
Tipo: <class 'numpy.ndarray'>
Shape: (450, 768)


**Recriar o indice FAISS**

In [38]:
import faiss

dimensao = embeddings.shape[1]

indice = faiss.IndexFlatIP(dimensao)

indice.add(embeddings)

print("Dimensão:", dimensao)
print("Vetores indexados:", indice.ntotal)

Dimensão: 768
Vetores indexados: 450


Criar a nova funcao de busca

In [39]:
def buscar_semanticamente(pergunta, top_k=5):

    embedding_pergunta = gerar_embeddings(
        pergunta,
        tipo="query"
    )

    scores, indices = indice.search(
        embedding_pergunta,
        top_k
    )

    resultados = []

    for score, idx in zip(scores[0], indices[0]):

        resultados.append({
            "score": float(score),
            "chunk_id": int(idx),
            "pagina": chunks[idx]["pagina"],
            "texto": chunks[idx]["texto"]
        })

    return resultados

Testar novamente

In [40]:
pergunta = "Como verificar o nível do óleo do motor?"

resultados = buscar_semanticamente(
    pergunta,
    top_k=5
)

for posicao, resultado in enumerate(resultados, start=1):

    print("=" * 100)

    print(
        f"TOP {posicao} | "
        f"Score: {resultado['score']:.4f} | "
        f"Página: {resultado['pagina']} | "
        f"Chunk: {resultado['chunk_id']}"
    )

    print()

    print(resultado["texto"][:800])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

TOP 1 | Score: 0.8935 | Página: 178 | Chunk: 294

vareta de medição até ao nível nominal. do nível de óleo. Verificar o nível de óleo do motor. ( 170) Para não sobrecarregar o Montar a vareta de medição meio ambiente desneces- do nível de óleo. sariamente, a BMW Motorrad recomenda que o óleo do mo- tor seja verificado pela mín. 50 km após viagens.
TOP 2 | Score: 0.8813 | Página: 178 | Chunk: 293

172 MANUTENÇÃO Reatestar óleo do motor Desligar a moto e colocá-la em posição de descanso, cer- tificando-se de que o piso é plano e firme. Limpar a zona do orifício de enchimento. Nível nominal do óleo do motor Entre a marca MIN e MAX Volume de reenchi- mento de óleo do motor máx. 0,5 l (Diferença entre Desmontar a vareta indica- MIN e MAX) dora do nível de óleo 1. Se o nível de óleo estiver abaixo da marca MIN: ATENÇÃO Atestar com óleo do motor. ( 172) Utilização de óleo do motor a menos ou a mais Se o nível de óleo estiver acima da marca MAX: Avaria do motor devido a en- Mandar corrigir o n

O TOP 1 e TOP 2 são diretamente relevantes.

O TOP 3 mostra uma limitação real: confundiu óleo do motor com óleo do travão devido à proximidade semântica.

O TOP 3 pode ser caracterizado como uma falha de recuperacao

**Interpretacao**

Foi identificado um caso real de falha do pipeline:

A primeira estratégia de extração do PDF apresentou perda de espaços entre palavras.

Embora o modelo de embeddings tenha recuperado trechos semanticamente relevantes, a baixa qualidade textual poderia prejudicar a etapa de geração.

A estratégia de extração foi ajustada utilizando palavras e coordenadas do PyMuPDF, preservando a separação lexical.


**Pipeline RAG**

Criar o contexto com os dados recuperados

In [41]:
def montar_contexto(resultados, limite_caracteres=4000):

    blocos = []

    total_caracteres = 0

    for resultado in resultados:

        bloco = (
            f"[PÁGINA {resultado['pagina']}]\n"
            f"{resultado['texto']}"
        )

        if total_caracteres + len(bloco) > limite_caracteres:
            break

        blocos.append(bloco)

        total_caracteres += len(bloco)

    contexto = "\n\n".join(blocos)

    return contexto

Testar o contexto

In [42]:
contexto = montar_contexto(resultados)

print(contexto)

[PÁGINA 178]
vareta de medição até ao nível nominal. do nível de óleo. Verificar o nível de óleo do motor. ( 170) Para não sobrecarregar o Montar a vareta de medição meio ambiente desneces- do nível de óleo. sariamente, a BMW Motorrad recomenda que o óleo do mo- tor seja verificado pela mín. 50 km após viagens.

[PÁGINA 178]
172 MANUTENÇÃO Reatestar óleo do motor Desligar a moto e colocá-la em posição de descanso, cer- tificando-se de que o piso é plano e firme. Limpar a zona do orifício de enchimento. Nível nominal do óleo do motor Entre a marca MIN e MAX Volume de reenchi- mento de óleo do motor máx. 0,5 l (Diferença entre Desmontar a vareta indica- MIN e MAX) dora do nível de óleo 1. Se o nível de óleo estiver abaixo da marca MIN: ATENÇÃO Atestar com óleo do motor. ( 172) Utilização de óleo do motor a menos ou a mais Se o nível de óleo estiver acima da marca MAX: Avaria do motor devido a en- Mandar corrigir o nível de chimento incorreto óleo numa oficina especiali- Prestar atenção a

**Carregando o Modelo FLAN-T5**

In [43]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

MODELO_LLM = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODELO_LLM
)

modelo_llm = AutoModelForSeq2SeqLM.from_pretrained(
    MODELO_LLM
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

modelo_llm = modelo_llm.to(DEVICE)

print("Modelo:", MODELO_LLM)
print("Dispositivo:", DEVICE)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Modelo: google/flan-t5-base
Dispositivo: cuda


**Criar a funcao de Geracao**

In [44]:
def gerar_resposta(prompt, max_new_tokens=200):

    entradas = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    entradas = {
        chave: valor.to(DEVICE)
        for chave, valor in entradas.items()
    }

    with torch.no_grad():

        saida = modelo_llm.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    resposta = tokenizer.decode(
        saida[0],
        skip_special_tokens=True
    )

    return resposta

**Criar a RAG**

**Objetivo** O manual da BMW F800GS possui centenas de páginas e milhares de palavras. Dessa forma, devido ao contexto e ao custo computacional, se torna inviavel enviar todo o documento para um LLM.

Assim, para resolver esse problema foi utilizada uma arquitetura RAG (Retrieval-Augmented Generation), abordagem na qual os trechos do manual são convertidos em vetores semânticos (embeddings).

Portanto, quando o usuário for realizar uma pergunta, o sistema vai procurar os trechos semanticamente mais próximos, enviando apenas esses trechos para o modelo de linguagem gerar a resposta.

**Arquitetura:**

Manual BMW F800GS (PDF) => Extração de texto => Chunking (divisão em trechos) => Embeddings (all-MiniLM-L6-v2) => FAISS (Base Vetorial) => Consulta do usuário => Embedding da consulta => Busca por similaridade => Top-K documentos => Flan-T5 => Resposta final

In [45]:
def responder_rag(pergunta, top_k=5):

    # 1. Recuperação
    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    # 2. Construção do contexto
    contexto = montar_contexto(
        resultados
    )

    # 3. Prompt aumentado
    prompt = f"""
Você é um assistente técnico especializado
no manual da motocicleta BMW F800GS.

Responda SOMENTE com base no CONTEXTO
DO MANUAL fornecido.

Não utilize conhecimento externo.

Se a resposta não estiver presente no contexto,
responda exatamente:

Não encontrei informação suficiente no manual recuperado.

Seja objetivo e responda em português.

PERGUNTA:

{pergunta}

CONTEXTO DO MANUAL:

{contexto}

RESPOSTA:
"""

    # 4. Geração
    resposta = gerar_resposta(
        prompt
    )

    return {
        "pergunta": pergunta,
        "resposta": resposta,
        "resultados": resultados
    }

Fazer a Primeira Pergunta RAG

In [46]:
resultado_rag = responder_rag(
    "Como verificar o nível do óleo do motor?"
)

print("PERGUNTA:")
print(resultado_rag["pergunta"])

print("\nRESPOSTA:")
print(resultado_rag["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do óleo do motor?

RESPOSTA:
[PGINA 178] vareta de mediço até a nvel nominal. do nvel de óleo. Verificar o nvel de óleo do motor. ( 170) Para no sobrecarregar o Montar a vareta de mediço meio ambiente desneces- do nvel de óleo. sariamente, a BMW Motorrad recomenda que o óleo do mo- tor seja verificado pela mn. 50 km após viagens. [PGINA 178] 172 MANUTENO Reatestar óleo do motor Desligar a moto e colocá-la em 


**O fluxo completo agora compreende**

Pergunta => multilingual-e5-base => Embedding da pergunta => FAISS => TOP 5 chunks => Montagem do contexto => Prompt aumentado => FLAN-T5 => Resposta fundamentada

Esse resultado mostra que o modelo está reproduzindo o contexto, em vez de responder à pergunta. Além disso, ainda existem resíduos da extração do PDF, como desneces- sariamente.

Para o projeto, isso é um caso de falha do RAG/geração, que sera corrigido com prompt engineering.

**Primeiro Confirmar o Tokenize**

In [47]:
print(type(tokenizer))
print(tokenizer.name_or_path)
print(type(modelo_llm))

<class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>
google/flan-t5-base
<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


Confirmado que o modelo do Tokenizer esta ok

**Melhorar o prompt do RAG**

Substituir responder.rag por esta versao, que em realidade seria mais curta e direta

In [48]:
def responder_rag(pergunta, top_k=3):

    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    contexto = montar_contexto(
        resultados,
        limite_caracteres=2500
    )

    prompt = f"""
Responda à pergunta usando somente o manual abaixo.

Pergunta:
{pergunta}

Manual:
{contexto}

Instruções:
- Responda diretamente à pergunta.
- Não copie o texto completo do manual.
- Não invente informações.
- Use português.
- Produza no máximo 5 frases.
- Se não encontrar a resposta, diga:
  "Não encontrei informação suficiente no manual recuperado."

Resposta:
"""

    resposta = gerar_resposta(
        prompt,
        max_new_tokens=150
    )

    return {
        "pergunta": pergunta,
        "resposta": resposta,
        "contexto": contexto,
        "resultados": resultados
    }

Usado Top3 porque a partir dai ja comecou a recuperar dados do oleo do travao, ou seja, informacao nao relevante para oleo do motor

**Testar novamente**

In [49]:
resultado_rag = responder_rag(
    "Como verificar o nível do óleo do motor?"
)

print("PERGUNTA:")
print(resultado_rag["pergunta"])

print("\nRESPOSTA:")
print(resultado_rag["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do óleo do motor?

RESPOSTA:
No encontrar a resposta, diga: "No encontrar a resposta suficiente no manual recuperado."


Como nao trouxe dados legiveis, significa que houve uma falha do FLAN-T5-base com o prompt atual: ele está copiando a instrução de fallback em vez de responder.

Sera corrigida removendo-se a frase que o modelo está copiando e reduzindo o contexto.

Substituindo responder.rag por esta versao

In [50]:
def responder_rag(pergunta, top_k=2):

    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    contexto = montar_contexto(
        resultados,
        limite_caracteres=1800
    )

    prompt = f"""
Tarefa: responda objetivamente à pergunta técnica.

Use exclusivamente as informações do trecho do manual.

Trecho do manual:
{contexto}

Pergunta:
{pergunta}

Escreva uma resposta curta em português.
Explique o procedimento encontrado no manual.
Não repita a pergunta.
Não copie as instruções desta tarefa.

Resposta:
"""

    resposta = gerar_resposta(
        prompt,
        max_new_tokens=120
    )

    return {
        "pergunta": pergunta,
        "resposta": resposta,
        "contexto": contexto,
        "resultados": resultados
    }

Testando novamente

In [51]:
resultado_rag = responder_rag(
    "Como verificar o nível do óleo do motor?"
)

print("PERGUNTA:")
print(resultado_rag["pergunta"])

print("\nRESPOSTA:")
print(resultado_rag["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do óleo do motor?

RESPOSTA:
Escreva uma resposta curta em português. Ejemplifica o procedimento encontrado no manual. No re- ta a questo. No copie as instruçes de te- fa.


A Resposta ainda continua ruim. Vamos testar o modelo diretamente

In [52]:
prompt_teste = """
Responda em português.

Informação:
O nível nominal do óleo do motor deve estar entre
as marcas MIN e MAX.

Pergunta:
Onde deve estar o nível do óleo do motor?

Resposta:
"""

resposta = gerar_resposta(
    prompt_teste,
    max_new_tokens=50
)

print(resposta)

MIN e MAX


**Prompt V1:** o modelo apresentou comportamento de cópia parcial do contexto recuperado e baixa capacidade de síntese.

**Prompt V2:** o prompt foi simplificado, o número de chunks foi reduzido de cinco para três e foram incluídas restrições explícitas de resposta direta, limite de cinco frases e proibição de cópia integral do contexto.

A resposta eh curta, mas semanticamente correta. Entretanto, o resultado demonstrou que o flan-t5-base apresenta limitacoes para instruções técnicas em português. Vamos utilizar outra alternativa

**Interpretacao**

O problema anterior é o prompt RAG muito complexo para o FLAN-T5-base, o que nao significa que o modelo esta quebrado.

Como melhoria a ideia seria fazer uma Versão 3 do prompt, usando few-shot prompting.

**Prompt RAG V3 - Few-shot**

In [53]:
def responder_rag_v3(pergunta, top_k=2):

    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    contexto = montar_contexto(
        resultados,
        limite_caracteres=1800
    )

    prompt = f"""
Responda perguntas sobre manutenção de motocicletas.

Exemplo:

Contexto:
A pressão dos pneus deve ser verificada regularmente.

Pergunta:
O que deve ser verificado nos pneus?

Resposta:
A pressão dos pneus deve ser verificada regularmente.

Agora responda usando o manual.

Contexto:
{contexto}

Pergunta:
{pergunta}

Resposta objetiva em português:
"""

    resposta = gerar_resposta(
        prompt,
        max_new_tokens=120
    )

    return {
        "pergunta": pergunta,
        "resposta": resposta,
        "contexto": contexto,
        "resultados": resultados
    }

Testando

In [54]:
resultado_v3 = responder_rag_v3(
    "Como verificar o nível do óleo do motor?"
)

print("PERGUNTA:")
print(resultado_v3["pergunta"])

print("\nRESPOSTA:")
print(resultado_v3["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do óleo do motor?

RESPOSTA:
[PGINA 178] 172 MANUTENO Reatestar óleo do motor Desligar a moto e colocá-la em posiço de descanso, cer- tificando-se de que o piso é plano e firme. Limpar a zona do orifcio de enchimento. Nvel nominal do óleo do motor Entre a marca MIN e MAX


A resposta com o Few-shot foi semanticamente mais completa

In [55]:
import pandas as pd

print("Pandas carregado:", pd.__version__)

Pandas carregado: 2.2.2


In [56]:
comparacao_prompts = pd.DataFrame([
    {
        "versao": "Prompt V1",
        "tecnica": "Role + Contexto + Regras",
        "resultado": "Copiou parcialmente o contexto"
    },
    {
        "versao": "Prompt V2",
        "tecnica": "Zero-shot com restrições",
        "resultado": "Copiou a instrução de fallback"
    },
    {
        "versao": "Prompt V3",
        "tecnica": "Few-shot",
        "resultado": resultado_v3["resposta"]
    }
])

comparacao_prompts

,versao,tecnica,resultado
0,Prompt V1,Role + Contexto + Regras,Copiou parcialmente o contexto
1,Prompt V2,Zero-shot com restrições,Copiou a instrução de fallback
2,Prompt V3,Few-shot,[PGINA 178] 172 MANUTENO Reatestar óleo do mot...


**Interpretacao:**

A aplicação de instruções complexas ao FLAN-T5-base apresentou baixa aderência ao formato esperado.

A simplificação do prompt e o uso de few-shot prompting melhoraram a orientação da geração.

O experimento demonstra que a qualidade do pipeline RAG não depende apenas da recuperação semântica, mas também da compatibilidade entre a estratégia de prompting e a capacidade do modelo gerador.

A resposta confirma que o FLAN-T5-base está fazendo extração/cópia do trecho, e não uma boa geração de resposta RAG em português.

A recuperação está funcionando muito bem. O problema está na etapa generativa.

A recomendação seria trocar apenas o LLM por Qwen2.5-1, ao passo que Embeddings e FAISS permaneceriam iguais.

**Liberar o Flan-T5 da memoria**

In [57]:
import gc
import torch

del modelo_llm
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memória liberada")

Memória liberada


**Carregar o Qwen**

In [58]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODELO_LLM
)

modelo_llm = AutoModelForCausalLM.from_pretrained(
    MODELO_LLM,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

print("Modelo carregado:", MODELO_LLM)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado: Qwen/Qwen2.5-1.5B-Instruct


**Criar nova funcao de Geracao**

Pelo fato de o Qwen ser decoder-only, a geracao se torna diferente do FLAN-T5

In [59]:
def gerar_resposta_qwen(prompt, max_new_tokens=200):

    mensagens = [
        {
            "role": "system",
            "content": (
                "Você é um assistente técnico especializado "
                "em manuais de motocicletas."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    texto_prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto_prompt,
        return_tensors="pt"
    ).to(modelo_llm.device)

    with torch.no_grad():

        saida = modelo_llm.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    novos_tokens = saida[
        0,
        entradas["input_ids"].shape[1]:
    ]

    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return resposta.strip()

**Testar o Qwen isoladamente, antes do RAG**

---



In [60]:
prompt_teste = """
Informação do manual:

O nível nominal do óleo do motor deve estar
entre as marcas MIN e MAX.

Pergunta:

Onde deve estar o nível do óleo do motor?

Responda em português e em uma frase.
"""

resposta = gerar_resposta_qwen(
    prompt_teste
)

print(resposta)

O nível ideal do óleo no motor está entre as marcações MIN e MAX indicadas no manual.


Essa troca se torna tecnicamente boa para o projeto, uma vez que com isso se pode ter uma comparacao das arquiteturas.

Dessa forma, temos:

*   O FLAN-T5 eh uma arquitetura Encoder-Decoder, cujo resultado apresenta tendencia a extracao / copia
*   O Qwen 2.5 apresenta arquitetura de Decoder-only, cujo resultado seria para geracao instrucional
*   O ES seria arquitetura somente de Encoder, para embeddings semanticos.

Isso permitiu discutir encoder-only versus decoder-only, bem como adequação ao caso de uso.




Como o Qwen está funcionando corretamente e gerando uma resposta natural em português, o proximo passo seria conectar o Qwen ao pipeline RAG já criado.

**Criar o RAG com Qwen**

In [61]:
def responder_rag_qwen(pergunta, top_k=3):

    # 1. Busca semântica
    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    # 2. Montagem do contexto
    contexto = montar_contexto(
        resultados,
        limite_caracteres=3000
    )

    # 3. Prompt aumentado
    prompt = f"""
Use exclusivamente os trechos do manual BMW F800GS
fornecidos abaixo para responder à pergunta.

CONTEXTO DO MANUAL:

{contexto}

PERGUNTA:

{pergunta}

INSTRUÇÕES:

1. Responda em português.
2. Responda diretamente à pergunta.
3. Não invente informações.
4. Não utilize conhecimento externo.
5. Não copie integralmente o contexto.
6. Produza uma resposta técnica e objetiva.
7. Se possível, explique o procedimento em etapas.

RESPOSTA:
"""

    # 4. Geração
    resposta = gerar_resposta_qwen(
        prompt,
        max_new_tokens=250
    )

    return {
        "pergunta": pergunta,
        "resposta": resposta,
        "contexto": contexto,
        "fontes": [
            {
                "pagina": resultado["pagina"],
                "score": resultado["score"],
                "chunk_id": resultado["chunk_id"]
            }
            for resultado in resultados
        ]
    }

**Executar a Primeira Consulta RAG**

In [62]:
resultado_rag = responder_rag_qwen(
    "Como verificar o nível do óleo do motor?"
)

print("PERGUNTA:")
print(resultado_rag["pergunta"])

print("\nRESPOSTA:")
print(resultado_rag["resposta"])

print("\nFONTES RECUPERADAS:")

for fonte in resultado_rag["fontes"]:
    print(
        f"Página: {fonte['pagina']} | "
        f"Score: {fonte['score']:.4f} | "
        f"Chunk: {fonte['chunk_id']}"
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do óleo do motor?

RESPOSTA:
Para verificar o nível do óleo do motor do BMW F800GS, siga essas etapas:

1. **Desligue a moto**: Certifique-se de que a moto esteja completamente desligada e parada em uma superfície plana e firme.

2. **Limpe a área**: Lave a região onde será feita a medição com água e detergente neutro para evitar contaminar o nível de óleo.

3. **Verifique a marca MIN e MAX**: Localize as marcas MIN e MAX na parte superior da vareta de medição. Essas marcas indicam o intervalo ideal para o nível de óleo.

4. **Monte a vareta de medição**: Arraste a vareta até que ela atinja a marca MIN. Isso significa que o nível atual está dentro do intervalo recomendado.

5. **Verifique se está abaixo da marca MIN**: Se o nível for abaixo dessa marca, você deve adicionar óleo conforme solicitado pelo manual (no caso, 0,5 litros).

6. **Monte novamente a vareta de medição**: Agora, mova-a até

FONTES RECUPERADAS:
Página: 178 | Score: 0.8935 | Chunk: 29

**Testar Outras Perguntas**

In [63]:
perguntas_teste = [
    "Como verificar o nível do óleo do motor?",
    "Qual deve ser a pressão dos pneus?",
    "Como verificar o nível do líquido de refrigeração?",
    "Como ajustar a corrente de transmissão?"
]

for pergunta in perguntas_teste:

    resultado = responder_rag_qwen(pergunta)

    print("=" * 100)
    print("PERGUNTA:")
    print(pergunta)

    print("\nRESPOSTA:")
    print(resultado["resposta"])

    print("\nFONTES:")

    for fonte in resultado["fontes"]:
        print(
            f"Página {fonte['pagina']} "
            f"| Score {fonte['score']:.4f}"
        )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do óleo do motor?

RESPOSTA:
Para verificar o nível do óleo do motor do BMW F800GS, siga essas etapas:

1. **Desligue a moto**: Certifique-se de que a moto esteja completamente desligada e parada em uma superfície plana e firme.

2. **Limpe a área**: Lave a região onde será feita a medição com água e detergente neutro para evitar contaminar o nível de óleo.

3. **Verifique a marca MIN e MAX**: Localize as marcas MIN e MAX na parte superior da vareta de medição. Essas marcas indicam o intervalo ideal para o nível de óleo.

4. **Monte a vareta de medição**: Arraste a vareta até que ela atinja a marca MIN. Isso significa que o nível atual está dentro do intervalo recomendado.

5. **Verifique se está abaixo da marca MIN**: Se o nível for abaixo dessa marca, você deve adicionar óleo conforme solicitado pelo manual (no caso, 0,5 litros).

6. **Monte novamente a vareta de medição**: Agora, mova-a até

FONTES:
Página 178 | Score 0.8935
Página 178 | Score 0.8813

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Qual deve ser a pressão dos pneus?

RESPOSTA:
Para determinar a pressão correta dos pneus do BMW F800GS, siga estes passos:

1. **Verifique a Tabela de Pressão**: Consulte a tabela fornecida no manual (Página 61), onde estão especificadas as pressões nominais dos pneus para diferentes condições de uso. Isso inclui velocidades, temperaturas e tipos de rodovias.

2. **Observar a Formação Térmica**: No painel de instrumentos, observe a formação térmica dos pneus. Esta formação indica como os pneus estão adaptando às condições climáticas e ao nível de carga.

3. **Adaptar a Pressão**: Baseando-se na formação térmica e na tabela de pressão, ajuste a pressão dos pneus conforme necessário. Evite valores muito altos ou baixos, pois isso pode afetar a eficiência do motor e a segurança do condutor.

4. **Verificação Regular**: Considere verificar a pressão dos pneus regularmente durante a viagem, especialmente após mudanças significativas em condições climáticas ou rodovias.

Lembre

F

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como verificar o nível do líquido de refrigeração?

RESPOSTA:
Para verificar o nível do líquido de refrigeração no BMW F800GS, siga essas etapas:

1. Abra o fecho do radiador de refrigeração, evitando queimar o motor.
2. Verifique o nível do líquido de refrigeração entre as marcas MIN e MAX no depósito de compensação.
3. Se o nível do líquido está abaixo do nível nominal, use um funil adequado para reatestar o líquido até ao nível nominal.
4. Certifique-se de que o piso onde você está trabalhando é plano e firme antes de começar qualquer trabalho mecânico.
5. Após reatestar o líquido, feche novamente o fecho do radiador e guarde cuidadosamente o funil usado.

FONTES:
Página 186 | Score 0.8718
Página 185 | Score 0.8703
Página 185 | Score 0.8660


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PERGUNTA:
Como ajustar a corrente de transmissão?

RESPOSTA:
Para ajustar a corrente de transmissão do BMW F800GS, siga as seguintes etapas:

1. **Verifique a Tensão**: Primeiro, verifique a tensão da corrente usando um instrumento adequado. O manual sugere verificar a tensão quando a corrente está no centro, entre a anilha e o carreto, para garantir que esteja dentro da tolerância autorizada.

2. **Desligue a Moto**: Quando você estiver pronto para fazer o ajuste, desligue a moto e coloque-a em posição de descanso, certificando-se de que o piso é plano e firme.

3. **Aperte a Porca do Eixo 1**: Use um binário para apertar a porca do eixo 1, que está localizada no braço oscillante do eixo de encaixe da roda traseira. Estebra 30...40 mm (veículo sem lante carga sobre o descanso lateral).

4. **Ajuste o Parafuso de Ajuste 1**: Após apertar a porca, use um parafuso

FONTES:
Página 202 | Score 0.8555
Página 127 | Score 0.8452
Página 202 | Score 0.8450


**Agora o Pipeline completo ficou assim**:

Manual BMW F800GS => PyMuPDF => Chunking com overlap => multilingual-e5-base => Embeddings normalizados => FAISS IndexFlatIP => Pergunta => Embedding query => Top-K chunks => Contexto aumentado => Qwen2.5-1.5B-Instruct => Resposta fundamentada

**Interpretacao**

O FLAN-T5-base funcionou em perguntas simples, mas apresentou tendência a copiar trechos do contexto no RAG; o Qwen2.5 mostrou melhor aderência a instruções e geração técnica em português.

A resposta ficou mais natural, mas foram identificadas alucinações importantes. Por exemplo, o contexto recuperado não dizia para:

lavar com água e detergente;
arrastar a vareta até a marca MIN;
adicionar exatamente 0,5 litro — o manual indica máximo de 0,5 l entre MIN e MAX.

Portanto, pode-se considerar evidencia de que o RAG reduz, mas não elimina alucinação.

Antes de se realizar as saidas controladas, o proximo passo seria criar um prompt RAG mais restritivo (exemplo top_k=2)



In [64]:
def responder_rag_qwen_v2(pergunta, top_k=2):

    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    contexto = montar_contexto(
        resultados,
        limite_caracteres=2200
    )

    prompt = f"""
Você é um assistente de consulta ao manual BMW F800GS.

REGRAS OBRIGATÓRIAS:
- Use SOMENTE informações explicitamente escritas no contexto.
- Não complete procedimentos com conhecimento próprio.
- Não crie etapas.
- Não altere valores ou unidades.
- Diferencie "máximo de 0,5 l" de "adicionar 0,5 l".
- Se uma informação não estiver no contexto, não a mencione.

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta}

TAREFA:
Responda objetivamente em português.
Liste apenas procedimentos explicitamente presentes no contexto.
Ao final informe as páginas utilizadas.

PÁGINAS DISPONÍVEIS:
{[r["pagina"] for r in resultados]}

RESPOSTA:
"""

    resposta = gerar_resposta_qwen(
        prompt,
        max_new_tokens=220
    )

    return {
        "pergunta": pergunta,
        "resposta": resposta,
        "contexto": contexto,
        "resultados": resultados
    }

**Teste**

In [65]:
resultado_v2 = responder_rag_qwen_v2(
    "Como verificar o nível do óleo do motor?"
)

print(resultado_v2["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Verifique o nível do óleo do motor:

1. Desligue a moto e coloque-a em posição de descanso, certificando-se de que o piso é plano e firme.
2. Limpe a zona do orifício de enchimento.
3. Insira a vareta de medição na válvula de combustão.
4. Marque o nível máximo do óleo no indicador.
5. Compare o nível marcado com o nível mínimo (MIN).
6. Se o nível for menor que MIN, adicione óleo conforme recomendado (0,5 litro).
7. Após adicionar óleo, desça novamente a vareta para marcar o novo nível.
8. Compare o novo nível com o máximo (MAX). Se for maior que MAX, houve sobrecarga; corrija o nível adicionando óleo se necessário.
9. Finalize ajustando o nível do óleo para o intervalo entre MIN e MAX.

PÁGINAS UTILIZADAS:
[178


**Comparando V1 e V2**

In [66]:
comparacao_rag = pd.DataFrame([
    {
        "versao": "RAG Qwen V1",
        "prompt": "Instruções gerais",
        "problema": (
            "Criou etapas não presentes no manual: "
            "água, detergente e movimentação da vareta"
        )
    },
    {
        "versao": "RAG Qwen V2",
        "prompt": "Grounding restritivo",
        "problema": resultado_v2["resposta"]
    }
])

comparacao_rag

,versao,prompt,problema
0,RAG Qwen V1,Instruções gerais,"Criou etapas não presentes no manual: água, de..."
1,RAG Qwen V2,Grounding restritivo,Verifique o nível do óleo do motor:\n\n1. Desl...


Na primeira versão do pipeline RAG com Qwen2.5-1.5B-Instruct, o modelo produziu uma resposta linguisticamente adequada, porém introduziu procedimentos não presentes nos trechos recuperados, como lavagem com água e detergente.

Também interpretou incorretamente o volume máximo de reenchimento de 0,5 l como uma instrução para adicionar exatamente esse volume.

O prompt foi revisado com regras explícitas de grounding, proibindo a inclusão de etapas não presentes no contexto e a alteração semântica de valores técnicos.

Esse é um ótimo caso de alucinação reduzida por engenharia de prompt, exatamente o que foi apresentado durante as aulas pelo Professor, durante as aulas.

**Saidas Controladas**

**JSON**

Definir Estrutura de Saida

In [67]:
from pydantic import BaseModel, Field
from typing import List


class RespostaTecnica(BaseModel):
    pergunta: str
    resposta: str
    procedimento: List[str]
    paginas: List[int]
    confianca: str = Field(
        pattern="^(alta|media|baixa)$"
    )

**Criar o prompt para o JSON**

In [68]:
def criar_prompt_json(pergunta, contexto, paginas):

    return f"""
Você é um assistente técnico do manual BMW F800GS.

Use SOMENTE informações presentes no contexto.

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta}

Retorne SOMENTE um JSON válido.

Use exatamente esta estrutura:

{{
  "pergunta": "texto",
  "resposta": "texto",
  "procedimento": [
    "etapa 1",
    "etapa 2"
  ],
  "paginas": [178],
  "confianca": "alta"
}}

REGRAS:
- Não use markdown.
- Não escreva texto antes do JSON.
- Não escreva texto depois do JSON.
- Não invente procedimentos.
- confiança deve ser: alta, media ou baixa.
- Use apenas páginas fornecidas: {paginas}.

JSON:
"""

**Criar Funcao para Extrair o JSON**

In [69]:
import json
import re


def extrair_json(texto):

    try:

        inicio = texto.find("{")
        fim = texto.rfind("}") + 1

        if inicio == -1 or fim == 0:
            raise ValueError(
                "Nenhum objeto JSON encontrado"
            )

        texto_json = texto[inicio:fim]

        return json.loads(texto_json)

    except json.JSONDecodeError as erro:

        print("Erro no parsing JSON:")
        print(erro)

        return None

**Validar com Pydantic**

In [70]:
def validar_resposta_json(dados):

    if dados is None:
        return None

    try:

        resposta_validada = RespostaTecnica(
            **dados
        )

        return resposta_validada

    except Exception as erro:

        print("Erro de validação:")

        print(erro)

        return None

**Criar o Pipeline RAG com Saida Controlada**

In [71]:
def responder_rag_json(pergunta, top_k=2):

    # Busca vetorial
    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    # Montar contexto
    contexto = montar_contexto(
        resultados,
        limite_caracteres=2200
    )

    paginas = list(set(
        resultado["pagina"]
        for resultado in resultados
    ))

    # Criar prompt
    prompt = criar_prompt_json(
        pergunta,
        contexto,
        paginas
    )

    # Gerar resposta
    resposta_llm = gerar_resposta_qwen(
        prompt,
        max_new_tokens=300
    )

    # Parsing JSON
    dados = extrair_json(
        resposta_llm
    )

    # Validação Pydantic
    resposta_validada = validar_resposta_json(
        dados
    )

    return {
        "resposta_bruta": resposta_llm,
        "json": dados,
        "validado": resposta_validada
    }

**Testar**

In [72]:
resultado_json = responder_rag_json(
    "Como verificar o nível do óleo do motor?"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

**Saida Original do Qwen**

In [73]:
print("RESPOSTA BRUTA:")

print(
    resultado_json["resposta_bruta"]
)

RESPOSTA BRUTA:
```json
{
  "pergunta": "Como verificar o nível do óleo do motor?",
  "resposta": "Verifique o nível do óleo do motor pelo orifício de enchimento. O nível ideal está entre a marca MIN e MAX.",
  "procedimento": ["Limpe a zona do orifício de enchimento.", "Verifique se o nível de óleo está entre a marca MIN e MAX.", "Se o nível for abaixo da marca MIN, ateste com óleo do motor.", "Se o nível for acima da marca MAX, correja o nível de óleo para dentro do intervalo mínimo e máximo permitidos."],
  "paginas": [178],
  "confianca": "alta"
}
```


**Saida do JSON**

In [74]:
print("\nOBJETO VALIDADO:")

print(
    resultado_json["validado"]
)


OBJETO VALIDADO:
pergunta='Como verificar o nível do óleo do motor?' resposta='Verifique o nível do óleo do motor pelo orifício de enchimento. O nível ideal está entre a marca MIN e MAX.' procedimento=['Limpe a zona do orifício de enchimento.', 'Verifique se o nível de óleo está entre a marca MIN e MAX.', 'Se o nível for abaixo da marca MIN, ateste com óleo do motor.', 'Se o nível for acima da marca MAX, correja o nível de óleo para dentro do intervalo mínimo e máximo permitidos.'] paginas=[178] confianca='alta'


**Prompt com Papel + Contexto + Tarefa + Formato**


Tal tecnica se torna uma das mais utilizadas atualmente

In [75]:
def criar_prompt_pctf(pergunta, contexto):

    prompt = f"""
PAPEL:
Você é um assistente técnico especializado no manual
da motocicleta BMW F800GS.

CONTEXTO:
Você deve utilizar exclusivamente os trechos do manual
fornecidos abaixo.

{contexto}

TAREFA:
Responda à seguinte pergunta técnica:

{pergunta}

Analise somente as informações presentes no contexto.
Não utilize conhecimento externo.
Não invente procedimentos, valores ou componentes.

FORMATO:
Retorne a resposta utilizando exatamente a estrutura:

RESPOSTA: <resposta técnica objetiva>

PROCEDIMENTO:
- <etapa encontrada no manual>
- <etapa encontrada no manual>

FONTE:
- Página do manual consultada

Se não houver procedimento explícito no contexto,
informe isso na resposta.

RESPOSTA FINAL:
"""

    return prompt

**Criar a funcao de execucao**

In [76]:
def responder_prompt_pctf(pergunta, top_k=2):

    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    contexto = montar_contexto(
        resultados,
        limite_caracteres=2200
    )

    prompt = criar_prompt_pctf(
        pergunta,
        contexto
    )

    resposta = gerar_resposta_qwen(
        prompt,
        max_new_tokens=250
    )

    return {
        "pergunta": pergunta,
        "prompt": prompt,
        "resposta": resposta,
        "resultados": resultados
    }

**Testar**

In [77]:
resultado_pctf = responder_prompt_pctf(
    "Como verificar o nível do óleo do motor?"
)

print(resultado_pctf["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RESPOSTA: Verifique o nível do óleo do motor pelo orifício de enchimento. Use uma vareta de medição para determinar se está dentro dos limites mínimo (MIN) e máximo (MAX). Caso o nível esteja abaixo da marca MIN, adicione óleo conforme recomendado. Caso esteja acima da marca MAX, remova excesso de óleo antes de montar novamente a vareta de medição. 

PROCEDIMENTO:
1. Verifique o nível do óleo do motor pelo orifício de enchimento.
2. Use uma vareta de medição para determinar se está dentro dos limites mínimo (MIN) e máximo (MAX).

FONTE:
- Página 178 do manual da motocicleta BMW F800GS


**O Prompt Usado:**

In [78]:
print(resultado_pctf["prompt"])


PAPEL:
Você é um assistente técnico especializado no manual
da motocicleta BMW F800GS.

CONTEXTO:
Você deve utilizar exclusivamente os trechos do manual
fornecidos abaixo.

[PÁGINA 178]
vareta de medição até ao nível nominal. do nível de óleo. Verificar o nível de óleo do motor. ( 170) Para não sobrecarregar o Montar a vareta de medição meio ambiente desneces- do nível de óleo. sariamente, a BMW Motorrad recomenda que o óleo do mo- tor seja verificado pela mín. 50 km após viagens.

[PÁGINA 178]
172 MANUTENÇÃO Reatestar óleo do motor Desligar a moto e colocá-la em posição de descanso, cer- tificando-se de que o piso é plano e firme. Limpar a zona do orifício de enchimento. Nível nominal do óleo do motor Entre a marca MIN e MAX Volume de reenchi- mento de óleo do motor máx. 0,5 l (Diferença entre Desmontar a vareta indica- MIN e MAX) dora do nível de óleo 1. Se o nível de óleo estiver abaixo da marca MIN: ATENÇÃO Atestar com óleo do motor. ( 172) Utilização de óleo do motor a menos ou a

O prompt PCTF separa explicitamente o papel do modelo, a base factual, a tarefa e o formato esperado, melhorando a previsibilidade. Entretanto, nao seria possivel tratar a resposta como necessariamente correta, ou seja, se torna importante validar cada procedimento realmente aparece nos chunks recuperados, porque já foram observadas alucinações no Qwen

**Zero Shot Prompt**

Para este caso, usando um prompt sem exemplos prévios. O modelo recebe apenas a tarefa e o contexto recuperado

In [79]:
def criar_prompt_zero_shot(pergunta, contexto):

    prompt = f"""
Use o contexto do manual BMW F800GS para responder
à pergunta técnica.

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta}

Responda objetivamente em português.
Não utilize informações externas ao contexto.

RESPOSTA:
"""

    return prompt

**Funcao de Execucao**

In [80]:
def responder_zero_shot(pergunta, top_k=2):

    # Recuperação dos chunks
    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    # Construção do contexto
    contexto = montar_contexto(
        resultados,
        limite_caracteres=2200
    )

    # Prompt Zero-Shot
    prompt = criar_prompt_zero_shot(
        pergunta,
        contexto
    )

    # Geração
    resposta = gerar_resposta_qwen(
        prompt,
        max_new_tokens=200
    )

    return {
        "pergunta": pergunta,
        "prompt": prompt,
        "resposta": resposta,
        "resultados": resultados
    }

**Testando o Zero-Shot**

In [81]:
resultado_zero_shot = responder_zero_shot(
    "Como verificar o nível do óleo do motor?"
)

print("PROMPT ZERO-SHOT:")
print(resultado_zero_shot["prompt"])

print("\nRESPOSTA:")
print(resultado_zero_shot["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PROMPT ZERO-SHOT:

Use o contexto do manual BMW F800GS para responder
à pergunta técnica.

CONTEXTO:
[PÁGINA 178]
vareta de medição até ao nível nominal. do nível de óleo. Verificar o nível de óleo do motor. ( 170) Para não sobrecarregar o Montar a vareta de medição meio ambiente desneces- do nível de óleo. sariamente, a BMW Motorrad recomenda que o óleo do mo- tor seja verificado pela mín. 50 km após viagens.

[PÁGINA 178]
172 MANUTENÇÃO Reatestar óleo do motor Desligar a moto e colocá-la em posição de descanso, cer- tificando-se de que o piso é plano e firme. Limpar a zona do orifício de enchimento. Nível nominal do óleo do motor Entre a marca MIN e MAX Volume de reenchi- mento de óleo do motor máx. 0,5 l (Diferença entre Desmontar a vareta indica- MIN e MAX) dora do nível de óleo 1. Se o nível de óleo estiver abaixo da marca MIN: ATENÇÃO Atestar com óleo do motor. ( 172) Utilização de óleo do motor a menos ou a mais Se o nível de óleo estiver acima da marca MAX: Avaria do motor devi

O Zero-shot prompting é uma técnica na qual o modelo executa uma tarefa sem receber exemplos prévios de perguntas e respostas.

Neste projeto, o modelo recebeu apenas o contexto recuperado do manual BMW F800GS, a pergunta do usuário e uma instrução para responder com base no contexto.

**Chain of Thought**

**Estrategia**

Implementar Chain-of-Thought (CoT) de forma controlada, solicitando ao modelo uma análise interna e retornando apenas uma justificativa técnica resumida. Isso evita depender da exposição do raciocínio detalhado do modelo

In [82]:
def criar_prompt_cot(pergunta, contexto):

    prompt = f"""
Você é um assistente técnico especializado no manual
da motocicleta BMW F800GS.

CONTEXTO DO MANUAL:
{contexto}

PERGUNTA:
{pergunta}

TAREFA:
Analise internamente o contexto antes de responder.

Durante a análise:
1. Identifique o assunto principal da pergunta.
2. Localize no contexto informações diretamente relacionadas.
3. Verifique valores, unidades e limites técnicos.
4. Ignore informações sobre outros componentes.
5. Verifique se o contexto é suficiente.
6. Formule uma resposta baseada somente no manual.

Não apresente o raciocínio interno detalhado.

FORMATO DA RESPOSTA:

RESPOSTA:
<resposta objetiva baseada no manual>

JUSTIFICATIVA:
<breve justificativa indicando qual informação do contexto
sustenta a resposta>

RESPOSTA FINAL:
"""

    return prompt

**Funcao para Executar o Prompt**

In [83]:
def responder_cot(pergunta, top_k=3):

    resultados = buscar_semanticamente(
        pergunta,
        top_k=top_k
    )

    contexto = montar_contexto(
        resultados,
        limite_caracteres=3000
    )

    prompt = criar_prompt_cot(
        pergunta,
        contexto
    )

    resposta = gerar_resposta_qwen(
        prompt,
        max_new_tokens=250
    )

    return {
        "pergunta": pergunta,
        "prompt": prompt,
        "resposta": resposta,
        "resultados": resultados
    }

**Testar**

In [84]:
resultado_cot = responder_cot(
    "Qual deve ser o nível do óleo do motor?"
)

print("PROMPT:")
print(resultado_cot["prompt"])

print("\nRESPOSTA:")
print(resultado_cot["resposta"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PROMPT:

Você é um assistente técnico especializado no manual
da motocicleta BMW F800GS.

CONTEXTO DO MANUAL:
[PÁGINA 178]
vareta de medição até ao nível nominal. do nível de óleo. Verificar o nível de óleo do motor. ( 170) Para não sobrecarregar o Montar a vareta de medição meio ambiente desneces- do nível de óleo. sariamente, a BMW Motorrad recomenda que o óleo do mo- tor seja verificado pela mín. 50 km após viagens.

[PÁGINA 178]
172 MANUTENÇÃO Reatestar óleo do motor Desligar a moto e colocá-la em posição de descanso, cer- tificando-se de que o piso é plano e firme. Limpar a zona do orifício de enchimento. Nível nominal do óleo do motor Entre a marca MIN e MAX Volume de reenchi- mento de óleo do motor máx. 0,5 l (Diferença entre Desmontar a vareta indica- MIN e MAX) dora do nível de óleo 1. Se o nível de óleo estiver abaixo da marca MIN: ATENÇÃO Atestar com óleo do motor. ( 172) Utilização de óleo do motor a menos ou a mais Se o nível de óleo estiver acima da marca MAX: Avaria do m

Foi testada uma variação controlada de Chain-of-Thought.

O modelo foi instruído a analisar previamente o assunto da pergunta, selecionar informações diretamente relacionadas, verificar valores e unidades e avaliar a suficiência do contexto.

O raciocínio interno detalhado não foi solicitado como saída.

A resposta apresenta apenas o resultado e uma justificativa técnica resumida, permitindo avaliar a fundamentação sem depender da exposição da cadeia interna de raciocínio.

**Comparacao entre os prompts:**


*   Zero-Shot: possui como caracteristicas nao se utilizar de exemplos
*   Few-Shot: Inclui exemplos de perguntas e respostas
*   Papel+Contexto+Tarefa+Formato: Prompt estruturado por funcao.





Comparacao entre os prompts

*   O Zero-shot tem como vantagem o fato de ser simples e rapido, mas tem como limitacao a possibilidade de gerar respostas genericas
*   O Few-Shot demonstra ser mais consistente, porem requer um prompt maior
*   O Papel-contexto-tarefa tem a vantagem de produzir respostas padronizadas, mas exige um bom projeto do prompt
*   *   O Chain of thougt possui como vantagem a explicacao passo-a-passo, entretanto nem sempre melhora com modelos menores

Em resumo, foram testadas as estratégias de Prompt Engineering.

O Zero-shot apresentou respostas rápidas, porém menos padronizadas.

O Few-shot aumentou a consistência ao fornecer exemplos de entrada e saída.

O prompt baseado em Papel + Contexto + Tarefa + Formato produziu as respostas mais organizadas e aderentes ao manual técnico.

Para procedimentos técnicos, empregou-se Chain-of-Thought, incentivando respostas em etapas.

Além disso, foi implementada saída estruturadas em JSON, listas e tabelas, com validação por meio do módulo json do Python, permitindo detectar e tratar respostas malformadas.

